# MeetScribe - Trascrizione riunioni con GPU

Pipeline completa su Google Colab: **diarization (pyannote) + trascrizione (Whisper)**.

**Prima di iniziare:**
1. `Runtime > Change runtime type > T4 GPU`
2. Metti il file audio in `Google Drive`, cartella `MyDrive/recordings/`
3. Aggiungi il token HuggingFace nei **Secrets** di Colab (icona chiave a sinistra) con nome `HF_TOKEN`
   - Il token si crea su https://huggingface.co/settings/tokens
   - Accetta le condizioni dei modelli pyannote (una tantum):
     https://hf.co/pyannote/segmentation-3.0 · https://hf.co/pyannote/speaker-diarization-3.1

Poi: **Runtime > Esegui tutte**.

## 1. Setup (installa MeetScribe + dipendenze GPU)

In [ ]:
import os

REPO_URL = 'https://github.com/andreaceruti/meet-scribe.git'
repo_dir = '/content/meet-scribe'
if not os.path.exists(repo_dir):
    !git clone {REPO_URL} {repo_dir}
else:
    !cd {repo_dir} && git pull
%cd {repo_dir}

# Installa il progetto. Include `hf-xet`: SERVE per scaricare i modelli pyannote 4.x
# (community-1) dallo storage Xet di HuggingFace. Senza, Colab ripiega sul
# "xet-bridge" che restituisce 403. Lo installo anche esplicitamente per sicurezza.
!pip install -q hatchling
!pip install -q -e .
!pip install -q hf_xet

# Timeout finito sui download HF (una connessione appesa viene ritentata).
# NB: NON impostiamo HF_HUB_DISABLE_XET: con hf_xet il download Xet nativo funziona.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"

print("\nCommit in uso:")
!git -C {repo_dir} log --oneline -1

import torch
print(f"\nGPU disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Token HuggingFace

In [ ]:
import os
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')   # Secrets di Colab (icona chiave a sinistra)
    print('Token caricato dai Secrets di Colab')
except Exception:
    token = 'hf_XXXXXXXXXXXXXXXX'       # <-- in alternativa incolla qui il token
    print('Token impostato manualmente')

os.environ['HUGGING_FACE_TOKEN'] = token
os.environ['HF_TOKEN'] = token

## 3. Carica l'audio da Google Drive

Metti il file in `MyDrive/recordings/` e scrivi il nome qui sotto.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

# Nome del file audio dentro /content/drive/MyDrive/recordings/
AUDIO_FILENAME = '3.fit lot avvale.wav'   # <-- cambia col nome del tuo file

src = Path('/content/drive/MyDrive/recordings') / AUDIO_FILENAME
recordings_dir = Path('recordings')
recordings_dir.mkdir(exist_ok=True)
audio_path = recordings_dir / src.name
shutil.copy(src, audio_path)
print(f'File pronto: {audio_path} ({audio_path.stat().st_size/1e6:.1f} MB)')

## 4. Modello Whisper + pre-download dei modelli

Scegli il modello Whisper e pre-scarica tutto **prima** della trascrizione, così
un eventuale problema di download si vede subito qui (con retry) invece che a metà pipeline.

- `large-v3-turbo` = migliore su GPU (~1.6 GB). Se il download è lento/instabile, usa `"medium"` o `"small"`.

In [ ]:
import os, re, time
from huggingface_hub import snapshot_download
from huggingface_hub.utils import HfHubHTTPError
from faster_whisper.utils import download_model

WHISPER_MODEL = "large-v3-turbo"   # <-- opzioni piu' leggere: "medium" / "small"
tok = os.environ.get("HUGGING_FACE_TOKEN") or os.environ.get("HF_TOKEN")

# --- imposta il modello Whisper nel config (preserva i commenti) ---
CONFIG_PATH = "config.yaml"
with open(CONFIG_PATH, encoding="utf-8") as f:
    cfg = f.read()
RE = re.compile(r'''^([ \t]*model:[ \t]*)(['\"]?)[^'\"#\r\n]*\2([ \t]*(?:\#.*)?)$''', re.MULTILINE)
assert len(RE.findall(cfg)) == 1, "riga 'model:' non trovata in config.yaml"
cfg = RE.sub(lambda m: f'{m.group(1)}\"{WHISPER_MODEL}\"{m.group(3)}', cfg)
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    f.write(cfg)
print(f'[config] whisper.model = \"{WHISPER_MODEL}\"')

def robusto(fn, what):
    for i in range(1, 6):
        try:
            return fn()
        except (HfHubHTTPError, OSError, TimeoutError, ConnectionError) as e:
            code = getattr(getattr(e, "response", None), "status_code", None)
            if code in (401, 403, 404):
                raise
            if i == 5:
                raise
            print(f"[{what}] tentativo {i}/5 fallito, riprovo tra {2**(i-1)}s...")
            time.sleep(2 ** (i - 1))

# --- pre-scarica i modelli pyannote (incluso community-1 via Xet nativo) ---
for repo in ["pyannote/segmentation-3.0",
             "pyannote/speaker-diarization-3.1",
             "pyannote/speaker-diarization-community-1"]:
    robusto(lambda r=repo: snapshot_download(r, token=tok), repo)
    print(f"[pyannote] {repo}  OK")

# --- pre-scarica il modello Whisper ---
robusto(lambda: download_model(WHISPER_MODEL), "whisper")
print(f"[whisper] {WHISPER_MODEL}  OK")

print("\nTutti i modelli sono in cache. Esegui la cella successiva.")

## 5. Esegui la trascrizione

In [ ]:
import glob, os
audio_files = glob.glob('recordings/*')
latest = max(audio_files, key=os.path.getmtime)
print(f"File audio: {latest}\n")
!python -m meet_scribe.main --input "{latest}"

## 6. Risultati

In [ ]:
from pathlib import Path
for txt in sorted(Path('output').glob('*.txt')):
    print(f'=== {txt.name} ===')
    print(txt.read_text(encoding='utf-8')[:3000])
    print('...\n')

In [ ]:
from google.colab import files
from pathlib import Path
for f in Path('output').iterdir():
    print(f'Download: {f.name}')
    files.download(str(f))